In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.fft import rfft, rfftfreq

In [ ]:
ROOT_DIR = "."  # 본인의 데이터 압축 해제 경로에 맞게 수정하세요.
DATASET_CONFIG = {
    "learning": {
        "input_dir": os.path.join(ROOT_DIR, "Learning_set"),
        "output_path": "learning_set_features.csv",
    },
    "test": {
        "input_dir": os.path.join(ROOT_DIR, "Test_set"),
        "output_path": "test_set_features.csv",
    },
    "full_test": {
        "input_dir": os.path.join(ROOT_DIR, "Full_Test_Set"),
        "output_path": "full_test_features.csv",
    },
}
TARGET_DATASETS = ("learning", "test")

# PDF 문서 기준 운전 조건 매핑
CONDITION_MAP = {
    "Bearing1": {"condition": "C1", "rpm": 1800, "load": 4000},
    "Bearing2": {"condition": "C2", "rpm": 1650, "load": 4200},
    "Bearing3": {"condition": "C3", "rpm": 1500, "load": 5000}
}

In [ ]:
def calculate_rms(x):
    return np.sqrt(np.mean(x**2))

def calculate_cf(x):
    rms_val = calculate_rms(x)
    if rms_val == 0:
        return 1
    return np.max(np.abs(x)) / rms_val

def calculate_time_features(signal):
    """시간 도메인 특징 추출"""
    p2p_val = np.max(signal) - np.min(signal)
    skew_val = skew(signal)
    kurt_val = kurtosis(signal, fisher=False)

    return [p2p_val, skew_val, kurt_val]

def calculate_spectral_features(signal, fs=25600):
    """
    주파수 도메인 특징 추출
    1. Location: Centroid Frequency
    2. Dispersion: Spectral Spread
    3. Complexity: Spectral Entropy
    4. Peakiness: Spectral Crest Factor
    5. Asymmetry: RMS to Centroid Ratio
    6. Sparsity: Spectral Gini Index
    """
    n = len(signal)
    
    # 기초 FFT 연산 (Amplitude Spectrum & Power Spectrum)
    fft_vals = np.abs(rfft(signal))
    fft_vals_sq = fft_vals ** 2
    freqs = rfftfreq(n, d=1/fs)
    
    sum_fft = np.sum(fft_vals)
    energy = np.sum(fft_vals_sq)
    
    dom_freq = freqs[np.argmax(fft_vals)] # 지배 주파수

    if sum_fft == 0 or energy == 0:
        return [dom_freq, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]

    centroid = np.dot(freqs, fft_vals) / sum_fft # 중심 주파수
    # Frequency Dispersion
    spectral_spread = np.sqrt(np.dot((freqs - centroid)**2, fft_vals) / sum_fft)
    
    # Spectral Entropy
    pdf = fft_vals / sum_fft
    entropy = -np.sum(pdf * np.log(pdf + 1e-9))
    
    spectral_crest = np.max(fft_vals) / (np.mean(fft_vals) + 1e-9)
    
    # RCR (RMS-to-Centroid Ratio)
    # RMS 주파수를 Centroid로 무차원화
    rms_numerator = np.dot(freqs ** 2, fft_vals_sq)
    rms_freq = np.sqrt(rms_numerator / energy)
    rcr = rms_freq / (centroid + 1e-9)
    
    # SGI (Spectral Gini Index)
    # 불평등도를 측정하는 지니계수 (정상=0 / 열화독점=1에 수렴)
    sorted_vals = np.sort(fft_vals)
    n_vals = len(sorted_vals)
    idx = np.arange(1, n_vals + 1)
    spectral_gini = np.sum((2 * idx - n_vals - 1) * sorted_vals) / (n_vals * sum_fft + 1e-9)
    
    return [
        dom_freq, 
        centroid, 
        spectral_spread, 
        entropy, 
        spectral_crest, 
        rcr, 
        spectral_gini
    ]

def read_sensor_file(path):
    with open(path, "r") as f:
        first_line = f.readline()

    sep = ';' if ';' in first_line else ','
    return pd.read_csv(path, header=None, sep=sep)

def vector_magnitude(h_signal, v_signal):
    return np.sqrt(h_signal**2 + v_signal**2)

def principal_axis_projection(h_signal, v_signal):
    X = np.column_stack([h_signal, v_signal])
    X = X - X.mean(axis=0)

    cov = np.cov(X, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argmax(eigvals)

    principal_vec = eigvecs[:, idx]
    principal_signal = X @ principal_vec

    # theta = np.arctan2(y, x): 주축 방향
    # axis_cos = np.cos(2*theta)
    axis_cos = ((principal_vec[0]**2 - principal_vec[1]**2)
                / (principal_vec[0]**2 + principal_vec[1]**2))

    # axis_sin = np.sin(2*theta)
    axis_sin = (2*(principal_vec[0] * principal_vec[1])
                / (principal_vec[0]**2 + principal_vec[1]**2))

    energy_ratio = eigvals[idx] / eigvals.sum()

    return principal_signal, axis_cos, axis_sin, energy_ratio

In [ ]:
def generate_features(dataset_name):
    config = DATASET_CONFIG[dataset_name]
    input_dir = config["input_dir"]
    output_path = config["output_path"]

    records = []
    bearing_folders = sorted(
        glob.glob(os.path.join(input_dir, "Bearing*_*"))
    )

    print(f"\n================================")
    print(f"Dataset: {dataset_name}")
    print(f"Input: {input_dir}")
    print(f"Output: {output_path}")
    print(f"================================")

    for folder in bearing_folders:
        bearing_name = os.path.basename(folder)

        cond_key = bearing_name.split("_")[0]
        cond_info = CONDITION_MAP[cond_key]

        print(f"\n[{bearing_name}] 처리 중...")

        # =====================================================
        # 1. Temperature Lookup Table 생성
        # =====================================================
        temp_lookup = {}
        temp_files = sorted(
            glob.glob(os.path.join(folder, "temp_*.csv"))
        )

        for temp_path in temp_files:
            try:
                temp_df = read_sensor_file(temp_path)

                hour = int(temp_df.iloc[0, 0])
                minute = int(temp_df.iloc[0, 1])

                temp_series = temp_df.iloc[:, 4]

                temp_lookup[(hour, minute)] = {
                    "temp_mean": temp_series.mean(),
                    "temp_max": temp_series.max(),
                    "temp_slope": (
                        temp_series.iloc[-1] - temp_series.iloc[0]
                    )
                }

            except Exception as e:
                print(
                    f"Temperature parsing 실패 "
                    f"({os.path.basename(temp_path)}): {e}"
                )

        # =====================================================
        # 2. Accelerometer Feature Extraction
        # =====================================================
        acc_files = sorted(
            glob.glob(os.path.join(folder, "acc_*.csv"))
        )
        print(f"ACC 파일 수: {len(acc_files)}")

        for file_path in acc_files:
            file_name = os.path.basename(file_path)

            try:
                file_idx = int(re.findall(r"\d+", file_name)[0])
                elapsed_sec = (file_idx - 1) * 10
                df = read_sensor_file(file_path)

                # ---------------------------------
                # timestamp 추출
                # ---------------------------------
                hour = int(df.iloc[0, 0])
                minute = int(df.iloc[0, 1])

                # ---------------------------------
                # signal 추출
                # ---------------------------------
                h_signal = df.iloc[:, 4].values
                v_signal = df.iloc[:, 5].values

                vector_signal = vector_magnitude(h_signal, v_signal)
                principal_signal, axis_cos, axis_sin, ratio = (
                    principal_axis_projection(h_signal, v_signal)
                )

                # ---------------------------------
                # feature 계산
                # ---------------------------------
                time_feat = calculate_time_features(principal_signal)
                freq_feat = calculate_spectral_features(principal_signal)

                rms_vector = calculate_rms(vector_signal)
                crest_factor = calculate_cf(vector_signal)

                # ---------------------------------
                # temperature lookup
                # ---------------------------------
                temp_info = temp_lookup.get((hour, minute), None)

                if temp_info is not None:
                    temp_mean = temp_info["temp_mean"]
                    temp_max = temp_info["temp_max"]
                    temp_slope = temp_info["temp_slope"]

                else:
                    temp_mean = np.nan
                    temp_max = np.nan
                    temp_slope = np.nan

                # ---------------------------------
                # record 생성
                # ---------------------------------
                record = {
                    "bearing": bearing_name,
                    "condition": cond_info["condition"],
                    "rpm": cond_info["rpm"],
                    "load": cond_info["load"],

                    "file_idx": file_idx,
                    "hour": hour,
                    "minute": minute,
                    "second": int(df.iloc[0, 2]),
                    "elapsed_sec": elapsed_sec,

                    "rms": rms_vector,
                    "cf": crest_factor,

                    "principal_axis_cos": axis_cos,
                    "principal_axis_sin": axis_sin,
                    "principal_energy_ratio": ratio,

                    "p2p": time_feat[0],
                    "skew": time_feat[1],
                    "kurt": time_feat[2],

                    "dom_freq": freq_feat[0],
                    "centroid_freq": freq_feat[1],
                    "spectral_spread": freq_feat[2],
                    "spectral_entropy": freq_feat[3],
                    "spectral_crest": freq_feat[4],
                    "rms2cent": freq_feat[5],
                    "spectral_gini": freq_feat[6],

                    "temp_mean": temp_mean,
                    "temp_max": temp_max,
                    "temp_slope": temp_slope
                }

                records.append(record)

            except Exception as e:
                print(f"오류 발생 ({file_name}): {e}")
                continue

    feature_df = pd.DataFrame(records)

    print("\n================================")
    print(f"Dataset: {dataset_name}")
    print(f"생성된 Row 수: {len(feature_df)}")

    feature_df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

    return feature_df

In [ ]:
for dataset_name in TARGET_DATASETS:
    if dataset_name not in DATASET_CONFIG:
        raise ValueError(
            f"알 수 없는 dataset: {dataset_name}. "
            f"사용 가능: {list(DATASET_CONFIG.keys())}"
        )

    generate_features(dataset_name)